# 🧟 SurviveCity GRPO Training — Colab Runner

Trains Qwen2.5-3B-Instruct with LoRA + GRPO on the SurviveCity multi-agent zombie survival environment.

**Meta × PyTorch × Scaler OpenEnv Hackathon** — Team PyGuys (Sirjan + Eeshan)

### Setup before you click **Runtime → Run All**

1. **Runtime → Change runtime type → T4 GPU** (free tier works; A100/L4 even better)
2. Click 🔑 **Secrets** in the left sidebar and add **two secrets**:
   - `GITHUB_TOKEN` — GitHub Personal Access Token (classic, `repo` scope) from https://github.com/settings/tokens
   - `HF_TOKEN` — HuggingFace token (**write** scope) from https://huggingface.co/settings/tokens
3. Edit `HUB_MODEL_ID` in the Configuration cell below to your HF username/repo

## 1. Configuration

In [ ]:
# -------- EDIT ME --------
HUB_MODEL_ID    = "sirjansingh/zombiee-qwen-grpo-lora"  # your HF user/repo
REPO_URL        = "https://github.com/SirjanSingh/zombiee.git"
REPO_BRANCH     = "master"
# -------------------------

# Model / training knobs — tuned for Colab T4 (16 GB).
MODEL_NAME       = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
MAX_STEPS        = 500          # enough to show improvement for judges; increase if A100
SAVE_STEPS       = 50           # frequent saves — Colab sessions can die
MAX_SEQ_LENGTH   = 2048
NUM_GENERATIONS  = 4            # GRPO group size; lower = less memory
LR               = 1e-5
LORA_R           = 16
LORA_ALPHA       = 32
OUTPUT_DIR       = "/content/lora_v1"
REPO_DIR         = "/content/zombiee"
ENV_PORT         = 7860

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("WANDB_DISABLED", "true")
print("Config loaded. HUB_MODEL_ID =", HUB_MODEL_ID)

## 2. GPU Sanity Check

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.free,compute_cap --format=csv

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Compute capability: {cap[0]}.{cap[1]}")
    print(f"bf16 native: {cap[0] >= 8}")
    free, total = torch.cuda.mem_get_info(0)
    print(f"Memory: {free/1e9:.1f}GB free / {total/1e9:.1f}GB total")

## 3. Install Dependencies

Pinned versions matching Dockerfile.dgx. The torchao 0.7.0 pin is critical for torch 2.4/2.5.

In [ ]:
%%bash
set -e
pip install -q --upgrade pip setuptools wheel

# Colab ships torch 2.x; we don't reinstall it
python -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda)"

# Core HF stack
pip install -q \
    "transformers==4.46.3" \
    "accelerate==1.1.1" \
    "peft==0.13.2" \
    "datasets==3.1.0" \
    "trl==0.12.1" \
    "bitsandbytes==0.44.1"

# Unsloth — install last with --no-deps to avoid bumping pinned versions
pip install -q --no-deps "unsloth @ git+https://github.com/unslothai/unsloth.git" "unsloth_zoo"

# torchao MUST be 0.7.x on torch 2.4/2.5 (newer references torch.int1 which is torch 2.6+)
pip install -q --force-reinstall --no-deps "torchao==0.7.0"

# Env server + utils
pip install -q "pydantic>=2.0" "fastapi>=0.104" "uvicorn[standard]>=0.24" \
    "matplotlib>=3.8" tensorboard "huggingface_hub>=0.25"

echo '--- import sanity check ---'
python - <<'PY'
import torch, torchao, transformers
from transformers.quantizers.auto import AutoHfQuantizer
print('torch', torch.__version__)
print('torchao', torchao.__version__)
print('transformers', transformers.__version__)
print('\u2705 Import chain OK')
PY

## 4. Load Secrets & Clone Private Repo

Reads `GITHUB_TOKEN` from Colab Secrets (🔑) to authenticate with the private GitHub repo.

In [ ]:
# ── Load GITHUB_TOKEN into env so git can use it ─────────────
try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    print("Loaded GITHUB_TOKEN from Colab Secrets")
except Exception:
    if not os.environ.get("GITHUB_TOKEN"):
        import getpass
        os.environ["GITHUB_TOKEN"] = getpass.getpass("Enter GitHub PAT: ")

import os, subprocess

def _run(cmd, **kwargs):
    """Run a command, print output, raise on failure."""
    print("$", " ".join(str(c) for c in cmd))
    result = subprocess.run(cmd, check=True, text=True,
                            capture_output=True, **kwargs)
    if result.stdout:
        print(result.stdout.rstrip())
    return result

# ── build authenticated clone URL ───────────────────────────────
# Inject token so git never prompts interactively.
clone_url = REPO_URL
gh_token  = os.environ.get("GITHUB_TOKEN", "")
if gh_token and clone_url.startswith("https://github.com/"):
    clone_url = clone_url.replace("https://", f"https://{gh_token}@")
    print("Auth: using GITHUB_TOKEN for clone.")
else:
    print("Auth: no token \u2014 public repo only.")

# Disable ALL interactive git prompts (belt-and-suspenders).
env = os.environ.copy()
env["GIT_TERMINAL_PROMPT"] = "0"
env["GIT_ASKPASS"]         = "echo"

# ── clone or update ─────────────────────────────────────────────
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo exists \u2014 pulling latest...")
    _run(["git", "-C", REPO_DIR, "fetch", "--all"],         env=env)
    _run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH],    env=env)
    _run(["git", "-C", REPO_DIR, "pull", "--ff-only"],      env=env)
else:
    _run(["git", "clone", "--branch", REPO_BRANCH,
          clone_url, REPO_DIR], env=env)

# Install as editable package
_run(["pip", "install", "-q", "-e", "."], cwd=REPO_DIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("\n--- repo head ---")
_run(["git", "-C", REPO_DIR, "log", "--oneline", "-3"])

# Clear token from local variables
del gh_token, clone_url
print("\u2705 Repo cloned and installed")

## 5. HuggingFace Hub Login

Uses the `HF_TOKEN` Colab secret for Hub push access.

In [ ]:
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    import getpass
    hf_token = getpass.getpass('Enter your HuggingFace token (write scope): ')

login(token=hf_token, add_to_git_credential=True)
os.environ['HUGGINGFACE_TOKEN'] = hf_token
os.environ['HF_TOKEN'] = hf_token

api = HfApi()
try:
    api.create_repo(repo_id=HUB_MODEL_ID, exist_ok=True, private=False)
    print(f'Hub repo ready: https://huggingface.co/{HUB_MODEL_ID}')
except Exception as e:
    print(f'create_repo warning (non-fatal if repo exists): {e}')

## 6. Detect Existing Checkpoints on Hub

If a previous run (Colab, Kaggle, or DGX) pushed checkpoints, we'll resume from them.

In [ ]:
resume_arg = []
try:
    files = api.list_repo_files(HUB_MODEL_ID)
    has_ckpt = any(f.startswith('checkpoint-') or f == 'trainer_state.json' for f in files)
    if has_ckpt:
        print(f'Found existing artifacts in {HUB_MODEL_ID} \u2192 will resume.')
        resume_arg = ['--resume-from-checkpoint', HUB_MODEL_ID]
    else:
        print(f'{HUB_MODEL_ID} is empty; starting fresh training.')
except Exception as e:
    print(f'Could not list repo (probably empty / new): {e}; starting fresh.')
print('resume_arg =', resume_arg)

## 7. Quick Env Smoke Test

Verify the environment runs correctly before starting training.

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from survivecity_env.env import SurviveCityEnv
import random

env = SurviveCityEnv()
obs = env.reset(seed=42)
print(f"Reset OK: step={obs['step_count']}, reward={obs['reward']:.4f}, done={obs['done']}")
assert 0.01 <= obs['reward'] <= 0.99, f"Reward {obs['reward']} out of bounds!"

# Run a quick episode with random actions
actions = ['move_up', 'move_down', 'move_left', 'move_right', 'eat', 'wait']
steps = 0
while not obs.get('done', False) and steps < 350:
    agent_id = obs.get('metadata', {}).get('current_agent_id', 0)
    step_count = obs.get('step_count', 0)
    if step_count == 50:
        action = {'agent_id': agent_id, 'action_type': 'vote_lockout', 'vote_target': random.choice([0,1,2])}
    else:
        action = {'agent_id': agent_id, 'action_type': random.choice(actions)}
    obs = env.step(action)
    assert 0.01 <= obs['reward'] <= 0.99, f"Reward {obs['reward']} out of bounds at step {steps}"
    steps += 1

meta = obs.get('metadata', {})
print(f"Episode complete: {steps} actions, step_count={obs['step_count']}, done={obs['done']}")
print(f"Final reward: {obs['reward']:.4f}, healthy_alive={meta.get('healthy_alive')}")
print(f"Postmortems: {len(meta.get('postmortems', []))}")
print('\u2705 Environment smoke test PASSED!')

## 8. Start Env Server in Background

Needed for eval.py (training uses local env directly now).

In [ ]:
import subprocess, time, urllib.request, sys

os.makedirs(OUTPUT_DIR, exist_ok=True)
env_log = open(f'{OUTPUT_DIR}/env_server.log', 'w')
env_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app',
     '--host', '127.0.0.1', '--port', str(ENV_PORT)],
    cwd=REPO_DIR, stdout=env_log, stderr=subprocess.STDOUT,
)

# Wait until the server is responding
for i in range(30):
    try:
        urllib.request.urlopen(f'http://127.0.0.1:{ENV_PORT}/health', timeout=2)
        print(f'\u2705 Env server up after {i+1}s (PID {env_proc.pid})')
        break
    except Exception:
        time.sleep(1)
else:
    env_log.flush()
    with open(f'{OUTPUT_DIR}/env_server.log') as f:
        print(f.read())
    raise RuntimeError('Env server did not start; check log above')

# Verify health endpoint
import json, urllib.request
resp = urllib.request.urlopen(f'http://127.0.0.1:{ENV_PORT}/health')
health = json.loads(resp.read())
print(f'Health: {health}')
assert health == {'status': 'healthy'}, f'Expected healthy, got {health}'

## 9. Run GRPO Training 🚀

This cell streams training output live. On Colab free tier (T4), expect:
- ~500 steps in ~3-4 hours
- Checkpoints pushed to HF Hub every 50 steps
- If session dies, re-run notebook → auto-resumes from Hub checkpoint

**Tip:** Use `Runtime → Run all` and keep the tab open, or upgrade to Colab Pro for longer sessions.

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'training.train',
    '--env-url', f'http://127.0.0.1:{ENV_PORT}',
    '--model-name', MODEL_NAME,
    '--max-steps', str(MAX_STEPS),
    '--save-steps', str(SAVE_STEPS),
    '--max-seq-length', str(MAX_SEQ_LENGTH),
    '--num-generations', str(NUM_GENERATIONS),
    '--lr', str(LR),
    '--lora-r', str(LORA_R),
    '--lora-alpha', str(LORA_ALPHA),
    '--output-dir', OUTPUT_DIR,
    '--report-to', 'tensorboard',
    '--push-to-hub',
    '--hub-model-id', HUB_MODEL_ID,
    '--save-total-limit', '3',
] + resume_arg

print('Launching:', ' '.join(cmd))
print('='*60)
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    print(f"\n{'='*60}")
    print(f'[Training exited with code {rc}]')
    if rc == 0:
        print('\u2705 Training completed successfully!')
    else:
        print('\u274c Training failed \u2014 check output above')
except KeyboardInterrupt:
    print('\nInterrupted; sending SIGTERM. Latest checkpoint is on the Hub.')
    proc.terminate()
    proc.wait()

## 10. Safety Net Upload

Re-uploads whatever is in OUTPUT_DIR in case push_to_hub glitched.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    print(f'Uploading {OUTPUT_DIR} \u2192 {HUB_MODEL_ID}')
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        commit_message='final upload from colab session',
        ignore_patterns=['_resume/*', 'env_server.log'],
    )
    print(f'\u2705 Done. Resume on any machine with --resume-from-checkpoint {HUB_MODEL_ID}')
else:
    print(f'No output to upload at {OUTPUT_DIR}')

## 11. Quick Evaluation — Baseline vs Trained

Runs 20 random-baseline episodes and 20 trained episodes, generates comparison plots.

In [ ]:
import subprocess, sys

eval_cmd = [
    sys.executable, '-m', 'training.eval',
    '--env-url', f'http://127.0.0.1:{ENV_PORT}',
    '--episodes', '20',
    '--lora-path', OUTPUT_DIR,
    '--plots-dir', f'{REPO_DIR}/plots',
]
proc = subprocess.run(eval_cmd, cwd=REPO_DIR, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print('STDERR:', proc.stderr)

# Display plots if generated
from IPython.display import Image, display
plots_dir = f'{REPO_DIR}/plots'
for fname in ['survival_rate.png', 'vote_accuracy.png', 'infected_detection.png']:
    path = os.path.join(plots_dir, fname)
    if os.path.exists(path):
        print(f'\n{fname}:')
        display(Image(filename=path, width=600))

## 12. TensorBoard (Optional)

View training curves in the notebook.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUTPUT_DIR}

## 13. Cleanup

In [ ]:
try:
    env_proc.terminate(); env_proc.wait(timeout=10)
    print('\u2705 Env server stopped.')
except Exception as e:
    print('env server cleanup:', e)